## LoRAs of the World Unite - Training SOTA DreamBooth LoRA with Pivotal Tuning 🧨

In this notebook, we show how to fine-tune [Stable Diffusion XL (SDXL)](https://huggingface.co/docs/diffusers/main/en/api/pipelines/stable_diffusion/stable_diffusion_xl) with [DreamBooth](https://huggingface.co/docs/diffusers/main/en/training/dreambooth) and [LoRA](https://huggingface.co/docs/diffusers/main/en/training/lora) using some of the most popular SOTA methods.

Learn more about the techniques used in this exmaple [here](https://huggingface.co/blog/sdxl_lora_advanced_script)

Let's get started 🧪

## Dataset 🐶

Let's get our training data! For this example, we'll download some images from the hub.

If you already have a dataset on the hub you wish to use, you can skip this part and go straight to: "Prep for training 💻" section, where you'll simply specify the dataset name.

If your images are saved locally, and/or you want to add BLIP generated captions, pick option 1 or 2 below.

**Option 2:** download example images from the hub

In [ ]:
from huggingface_hub import snapshot_download

local_dir = "./data/3d_icon"  # @param
dataset_to_download = "LinoyTsaban/3d_icon"  # @param
snapshot_download(
    dataset_to_download,
    local_dir=local_dir,
    repo_type="dataset",
    ignore_patterns=".gitattributes",
)

Preview the images:

In [ ]:
from PIL import Image


def image_grid(imgs, rows, cols, resize=256):
    assert len(imgs) == rows * cols

    if resize is not None:
        imgs = [img.resize((resize, resize)) for img in imgs]
    w, h = imgs[0].size
    grid = Image.new("RGB", size=(cols * w, rows * h))
    grid_w, grid_h = grid.size

    for i, img in enumerate(imgs):
        grid.paste(img, box=(i % cols * w, i // cols * h))
    return grid

In [ ]:
import glob

img_paths = f"{local_dir}/*.jpg"
imgs = [Image.open(path) for path in glob.glob(img_paths)]

num_imgs_to_preview = 5
image_grid(imgs[:num_imgs_to_preview], 1, num_imgs_to_preview)

### Generate custom captions with BLIP

Load BLIP2 to auto caption your images: 


**Note:** if you downloaded the `LinoyTsaban/3d_icon dataset` from the hub, you would find it already contains captions (generated with BLIP and prefixed with a token identifier) in the `metadata.jsonl` file
You can skip this part if you wish to train on that dataset using the existing captions. 

## Prep for training 💻

Initialize `accelerate`:

In [ ]:
!accelerate config default

## Train! 🔬

### `Diffusers` 🧨 Training loop

### hyperparameters 📐

1. **How to choose your hyperparams?** Check out this [blog post]() - where we explore and comapre different hyperparmas and configurations for different use cases, depending on your data and subject.     

2. **Make sure to add** `push_to_hub` so that the checkpoint is automatically pushed to the Hub and doesn't get lost. The `--push_to_hub` argument ensures that the trained checkpoints are automatically pushed to the Hugging Face Hub.

3. Some paramters that can help us with **compute** when doing DreamBooth with LoRA on a heavy pipeline like Stable Diffusion XL:

    * Gradient checkpointing (`--gradient_accumulation_steps`)
    * 8-bit Adam (`--use_8bit_adam`) - optional when using `--optimizer='AdamW'`, with `--optimizer='Prodigy'` this will be ignored
    * Mixed-precision training (`--mixed-precision="bf16"`)

### Launch training 🚀🚀🚀

**To allow for custom captions** we need to install the `datasets` library:
- Use `--caption_column` to specify name of the caption column in your dataset. 
    - In this example we used `"prompt"` to
       save our captions in the metadata file, change this according to your needs.

**Otherwise:** 
- you can skip the installation if you want to train soley with `--instance_prompt`.
  in that case, specify `--instance_data_dir` instead of `--dataset_name`

#### 🤗Pick a name for your Dreambooth LoRA fine-tuned model:🤗

This name will be used to save your model, so pick an informative name based on your chosen concept💡

In [ ]:
output_dir = "training_logs/3d-icon-sdxl-lora"  # @param

**Instance & Validation Prompt**

* `instance_prompt` - 
    * when custom captions are enabled this prompt is still used in case there are missing captions, as well as in the model's readme. 
    * If custom captions are not used, this prompt will be used as the caption for all training images. 
* `validation_prompt` - 
    * this prompt is used to generate images throughout the training process, this way you can see the models learning curve during training. 
    * you can also change `num_validation_images` (4 by default) and `validation_epochs` (50 by default) to control the amount images generated with the validation prompt, and the number of epochs between each dreambooth validation.  

In [ ]:
instance_prompt = "3d icon in the style of TOK"  # @param
validation_prompt = "a TOK icon of an astronaut riding a horse, in the style of TOK"  # @param

**Set your LoRA rank**

The rank of your LoRA is linked to its expressiveness.
The bigger the rank the closer we are to regular dreambooth, and in theory we have more expressive power (and heavier weights). 

For a very simple concept that you have a good high quality image set for (e.g. a pet, a generic object), a rank as low as 4 can be enough to get great results. We reccomend going between 8 and 64 depending on your concept and how much of a priortiy it is for you to keep the LoRA small or not.  

In [ ]:
rank = 8  # @param

In [ ]:
# Call the train.sh bash script from inside the notebook
!bash train.sh